In [1]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------- CONFIGURACIÓN ------------------- #
COLUMNAS_MODELO = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]

# ------------------- CARGA DE DATOS Y ARTEFACTOS ------------------- #
df = pd.read_csv("sale_properties_clustered.csv")
df = df.dropna()

# Encoding por media del precio
df["distrito_encoded"] = df["distrito"].map(df.groupby("distrito")["price_eur"].mean())

# Artefactos globales
with open("imputer_sale.pkl", "rb") as f:
    imputer = pickle.load(f)
with open("scaler_global_sale.pkl", "rb") as f:
    scaler = pickle.load(f)

# ------------------- PREPROCESADO ------------------- #
X = df[COLUMNAS_MODELO]
y = df["price_eur"]

X_imputado = imputer.transform(X)
X_scaled = scaler.transform(X_imputado)

# ------------------- DIVISIÓN Y ENTRENAMIENTO ------------------- #
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# ------------------- EVALUACIÓN ------------------- #
y_pred = model.predict(X_test).ravel()
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"✅ MAE en test: {mae:,.2f} EUR")
print(f"✅ R² en test: {r2:.4f}")

# ------------------- GUARDADO ------------------- #
model.save("nn_model_global_sale.keras")

with open("../model/artifacts/sale_model_rrnn.pkl", "wb") as f:
    pickle.dump(model, f)

metrics = {"mae": mae, "mse": mse, "r2": r2}
with open("nn_model_metrics_global_sale.pkl", "wb") as f:
    pickle.dump(metrics, f)


c:\Users\Marta\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 3398917160960.0000 - mae: 1202909.3750 - val_loss: 2931594887168.0000 - val_mae: 1131700.3750
Epoch 2/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3310809251840.0000 - mae: 1172756.8750 - val_loss: 2931370491904.0000 - val_mae: 1131638.2500
Epoch 3/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3603987693568.0000 - mae: 1205806.1250 - val_loss: 2930095947776.0000 - val_mae: 1131327.2500
Epoch 4/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3257671090176.0000 - mae: 1149638.3750 - val_loss: 2925829816320.0000 - val_mae: 1130348.2500
Epoch 5/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4125092216832.0000 - mae: 1272670.6250 - val_loss: 2915273539584.0000 - val_mae: 1128011.6250
Epoch 6/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3422769119232.0000 - mae: 1196337.2500 - val_loss: 2894633369600.0000 - val_mae: 1123505.1250
Epoch 7/200
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 4207552233472.0000 - ma

In [2]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf

# Columnas requeridas
COLUMNAS_MODELO = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]

def predecir_precio_vivienda_nn(nueva_vivienda: dict) -> float:
    """
    Predice el precio de una vivienda usando el modelo de red neuronal global.

    Args:
        nueva_vivienda (dict): Diccionario con las claves de COLUMNAS_MODELO

    Returns:
        float: Precio estimado en euros
    """
    # Convertir a DataFrame
    df_nueva = pd.DataFrame([nueva_vivienda])
    df_nueva = df_nueva.reindex(columns=COLUMNAS_MODELO)  # Asegura el orden correcto

    # Cargar artefactos
    with open("imputer_sale.pkl", "rb") as f:
        imputer = pickle.load(f)
    with open("scaler_global_sale.pkl", "rb") as f:
        scaler = pickle.load(f)
    
    # Cargar modelo
    model = tf.keras.models.load_model("nn_model_global_sale.keras")

    # Preprocesamiento
    X_imputado = imputer.transform(df_nueva)
    X_scaled = scaler.transform(X_imputado)

    # Predicción
    y_pred = model.predict(X_scaled).ravel()[0]

    print(f"🏠 Precio estimado (NN): {y_pred:,.2f} EUR")
    return y_pred


In [3]:
nueva_vivienda = {
    "superficie_construida": 120,
    "banos": 2,
    "distrito_encoded": 1800000,
    "habitaciones": 3,
    "planta_numerica": 2,
    "exterior": 1,
    "antiguedad": 15,
    "terraza": 1,
    "garaje": 1,
    "calefaccion": 1
}

precio_estimado = predecir_precio_vivienda_nn(nueva_vivienda)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
🏠 Precio estimado (NN): 2,407,929.00 EUR
